# Week 4 Live Session: Complete Production ML Pipeline

**Dataset:** Adult Income (continued from Week 3)

**Goal:** Build production-quality ML pipeline with:
- Cross-validation
- Hyperparameter tuning (GridSearchCV)
- Regularization
- Data leakage prevention (Pipeline)

**Scaffolding:** 50% (midpoint - equal partnership)

---

## Setup: Load Adult Income Dataset

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

%matplotlib inline

print("✅ All imports successful!")

✅ All imports successful!


In [15]:
# Load Adult Income dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nTarget distribution:")
print(df['income'].value_counts())

Dataset shape: (32561, 15)

First 3 rows:
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   

       marital-status         occupation   relationship   race   sex  \
0       Never-married       Adm-clerical  Not-in-family  White  Male   
1  Married-civ-spouse    Exec-managerial        Husband  White  Male   
2            Divorced  Handlers-cleaners  Not-in-family  White  Male   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0              13  United-States  <=50K  
2             0             0              40  United-States  <=50K  

Target distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


## Part 1: Data Preprocessing

Clean data and separate features from target.

In [16]:
# TODO: Drop rows with missing values
# Hint: Use dropna()
# Expected: DataFrame with no missing values

df_clean = df.dropna()

print(f"After cleaning: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

After cleaning: (32561, 15)
Missing values: 0


In [17]:
# Create binary target: 1 if income >50K, 0 otherwise
y = (df_clean['income'] == '>50K').astype(int)

# Drop target and irrelevant columns
X = df_clean.drop(['income', 'fnlwgt', 'education'], axis=1)

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nClass balance: {y.value_counts(normalize=True)}")

Features: (32561, 12)
Target: (32561,)

Class balance: income
0    0.75919
1    0.24081
Name: proportion, dtype: float64


## Part 2: Train/Test Split

Split data BEFORE any preprocessing to avoid leakage.

In [18]:
# TODO: Split data into train/test (80/20 split)
# Hint: train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# Expected: X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\n✅ Data split complete!")

Training samples: 26048
Test samples: 6513

✅ Data split complete!


### GridSearchCV and RandomizeSearchCV

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: Define parameter grid
param_grid = {
    'max_depth': [3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'n_estimators': [50, 100, 200]
}

# Step 2: Create base model
base_model = RandomForestClassifier(random_state=42)

# Step 3: Create GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

# Step 4: Fit (this does all the work)
grid_search.fit(X_train, y_train)

# Step 5: Get results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")
print(f"Test score: {grid_search.score(X_test, y_test):.3f}")


from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define distributions instead of lists
param_distributions = {
    'max_depth': randint(3, 20),           # Random int between 3 and 20
    'min_samples_split': randint(2, 20),
    'n_estimators': [50, 100, 200, 500],   # Can still use lists
    'max_features': uniform(0.1, 0.9)      # Random float between 0.1 and 1.0
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=50,  # Try 50 random combinations
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
print(f"Best params: {random_search.best_params_}")


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
Best CV score: 0.960
Test score: 0.956
Best params: {'max_depth': 8, 'max_features': np.float64(0.28714749658136995), 'min_samples_split': 5, 'n_estimators': 100}


### Regularization - Ridge (l2)

In [20]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Note: Using X_train, y_train from our Adult Income dataset
# Try different alpha values (regularization strength)
alphas = [0.001, 0.1, 1, 10, 100]
for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"Alpha={alpha:6.3f}: CV score = {scores.mean():.3f}")

Alpha= 0.001: CV score = 0.743
Alpha= 0.100: CV score = 0.744
Alpha= 1.000: CV score = 0.729
Alpha=10.000: CV score = 0.711
Alpha=100.000: CV score = 0.674


### Regularization - Lasso (l1)

In [21]:
from sklearn.linear_model import Lasso

model = Lasso(alpha=1.0)
model.fit(X_train, y_train)

# Show coefficient sparsity
print(f"Non-zero coefficients: {np.sum(model.coef_ != 0)} / {len(model.coef_)}")
print(f"Sparsity: {np.sum(model.coef_ == 0) / len(model.coef_) * 100:.1f}%")

# check accuaracy of the model
y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)  # Threshold at 0.5 for binary classification
accuracy = accuracy_score(y_test, y_pred_binary)
print(f"Test Accuracy: {accuracy:.3f}")

Non-zero coefficients: 3 / 30
Sparsity: 90.0%
Test Accuracy: 0.904


### Regularization - Elastic Net

In [22]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(alpha=1.0, l1_ratio=0.5)  # 50% L1, 50% L2
model.fit(X_train, y_train)

print(f"Non-zero coefficients: {np.sum(model.coef_ != 0)} / {len(model.coef_)}")
print(f"Sparsity: {np.sum(model.coef_ == 0) / len(model.coef_) * 100:.1f}%")

# check accuaracy of the model
y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)  # Threshold at 0.5 for binary classification
accuracy = accuracy_score(y_test, y_pred_binary)
print(f"Test Accuracy: {accuracy:.3f}")

Non-zero coefficients: 4 / 30
Sparsity: 86.7%
Test Accuracy: 0.939


### GridSearch with Regularization

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Try L1 and L2 with different alpha values
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Inverse of alpha (large C = weak regularization)
    'solver': ['saga'],  # Supports all penalty types
    'l1_ratio': [0.3, 0.5, 0.7]  # Only used for elasticnet
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print(f"Best regularization: {grid.best_params_['penalty']}")
print(f"Best C: {grid.best_params_['C']}")
print(f"CV score: {grid.best_score_:.3f}")

/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter 

Best regularization: elasticnet
Best C: 0.001
CV score: 0.923


/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Data Leakage

In [24]:
#BAD: Processing before splitting

from sklearn.impute import SimpleImputer

# BAD: Impute using ALL data's mean

#X_train, X_test, y_train, y_test = train_test_split(X_imputed, y)
#imputer = SimpleImputer(strategy='mean')
#X_imputed = imputer.fit_transform(X_train)  # ← Test set influences mean

In [25]:

pipe = Pipeline([
    ('scaler', StandardScaler()),      # Step 1: Scaling
    ('classifier', LogisticRegression(max_iter=1000))  # Step 2: Model
])

pipe.fit(X_train, y_train)
score = pipe.score(X_test, y_test)
print(f"Test score: {score:.3f}")

Test score: 0.982


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define transformers for different column types
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['age', 'hours_per_week']),
    ('cat', OneHotEncoder(), ['workclass', 'education'])
])

# Put in Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Use as before - no leakage!
pipe.fit(X_train, y_train)

ValueError: Specifying the columns using strings is only supported for dataframes.

## Part 3: Build ColumnTransformer

Handle numeric and categorical features separately.

In [ ]:
# Identify numeric and categorical features
numeric_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'marital-status', 'occupation', 'relationship', 
                        'race', 'sex', 'native-country']

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

Numeric features (5): ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Categorical features (7): ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [ ]:
# TODO: Create ColumnTransformer
# Hint: Use StandardScaler for numeric, OneHotEncoder for categorical
# Expected: ColumnTransformer object

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("✅ ColumnTransformer created!")

✅ ColumnTransformer created!


## Part 4: Build Pipeline

Combine preprocessing + model into single pipeline.

In [ ]:
# TODO: Create Pipeline with preprocessor + LogisticRegression
# Hint: Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=1000))])
# Expected: Pipeline object

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

print("✅ Pipeline created!")
print(pipeline)

✅ Pipeline created!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('classifier', LogisticRegression(max_iter=1000))])


## Part 5: Cross-Validation Evaluation

Get reliable performance estimate using CV.

In [ ]:
# TODO: Perform 5-fold stratified cross-validation
# Hint: cross_val_score(pipeline, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42))
# Expected: Array of 5 scores

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

print("Cross-Validation Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.3f}")

print(f"\n📊 CV Result: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/utils/_indexing.py", line 420, in _get_column_indices
    all_columns = X.columns
                  ^^^^^^^^^
AttributeError: 'numpy.ndarray' object has no attribute 'columns'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/lwgray/miniforge3/envs/data_science/lib/python3.11/site-packages/s

## Part 6: GridSearchCV - Hyperparameter Tuning

Find best hyperparameters automatically.

In [ ]:
# TODO: Define parameter grid
# Hint: Use 'classifier__C' and 'classifier__penalty' notation
# Expected: Dictionary with parameter ranges

param_grid = {
    'classifier__C': [0.001, 0.01, 0.1],  # Regularization strength
    'classifier__penalty': ['l2'],  # Use only 'l2' for default solver, or add solver parameter
    'classifier__solver': ['lbfgs']  # Specify solver that supports the penalties
}

# Alternative with L1 support:
# param_grid = {
#     'classifier__C': [0.001, 0.01, 0.1],
#     'classifier__penalty': ['l1', 'l2'],  
#     'classifier__solver': ['liblinear']  # Supports both l1 and l2
# }

print("Parameter grid:")
print(param_grid)

Parameter grid:
{'classifier__C': [0.001, 0.01, 0.1], 'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs']}


In [ ]:
# TODO: Create GridSearchCV
# Hint: GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# Expected: GridSearchCV object

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("✅ GridSearchCV created!")
print("\nFitting (this may take 1-2 minutes)...")

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

print("\n✅ GridSearchCV complete!")

✅ GridSearchCV created!

Fitting (this may take 1-2 minutes)...
Fitting 5 folds for each of 3 candidates, totalling 15 fits

✅ GridSearchCV complete!


In [ ]:
# Debug: Check data types
print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)
print("\nPreprocessor expects these columns:")
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

# Check if X (original data) is still available as DataFrame
try:
    print("\ndf_clean type:", type(df_clean))
    print("df_clean columns:", df_clean.columns.tolist())
except:
    print("\ndf_clean not available")

X_train type: <class 'numpy.ndarray'>
X_train shape: (455, 30)

Preprocessor expects these columns:
Numeric: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

df_clean type: <class 'pandas.core.frame.DataFrame'>
df_clean columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


In [ ]:
# Fix: Recreate train-test split with DataFrame for pipeline
from sklearn.model_selection import train_test_split

# Use the DataFrame from earlier (Adult Income dataset)
X_df = df_clean.drop(['income', 'fnlwgt', 'education'], axis=1)
y_df = df_clean['income']

# Create proper train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_df, test_size=0.2, stratify=y_df, random_state=42
)

print("✅ Fixed train-test split with DataFrame!")
print(f"X_train type: {type(X_train)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_train columns: {X_train.columns.tolist()}")

✅ Fixed train-test split with DataFrame!
X_train type: <class 'pandas.core.frame.DataFrame'>
X_train shape: (26048, 12)
X_train columns: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']


In [ ]:
# View results
print("="*60)
print("GRIDSEARCH RESULTS")
print("="*60)
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")

# Test set evaluation
test_score = grid_search.score(X_test, y_test)
print(f"\nTest set score: {test_score:.3f}")
print("="*60)

GRIDSEARCH RESULTS

Best parameters: {'classifier__C': 0.1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best CV score: 0.850

Test set score: 0.855


## Part 7: Save Pipeline

Save trained pipeline for production use.

In [ ]:
# TODO: Save best model using joblib
# Hint: joblib.dump(grid_search.best_estimator_, 'adult_income_pipeline.pkl')
# Expected: Saved .pkl file

joblib.dump(grid_search.best_estimator_, 'adult_income_pipeline.pkl')

print("✅ Pipeline saved as 'adult_income_pipeline.pkl'")

✅ Pipeline saved as 'adult_income_pipeline.pkl'


In [ ]:
# TODO: Load and test saved pipeline
# Hint: joblib.load('adult_income_pipeline.pkl')
# Expected: Loaded pipeline object

loaded_pipeline = joblib.load('adult_income_pipeline.pkl')

# Make predictions
test_predictions = loaded_pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Loaded pipeline test accuracy: {test_accuracy:.3f}")
print("✅ Pipeline loaded and tested successfully!")

Loaded pipeline test accuracy: 0.855
✅ Pipeline loaded and tested successfully!


## Summary: Production ML Checklist

✅ **Cross-Validation:** Used stratified 5-fold CV for reliable estimates

✅ **GridSearchCV:** Tuned hyperparameters systematically

✅ **Regularization:** Tested L1 and L2 penalties

✅ **Pipeline:** Prevented data leakage with proper preprocessing

✅ **Saved Model:** Ready for production deployment

**This is how professional data scientists work!**

---

*Next: Pair programming exercise to detect and fix data leakage*